# 04 — External / Enrichment Columns Review

**Source:** 33 columns NOT present in the raw transfer files or team stats — added during data processing  
**Goal:** Understand what each adds, check coverage, decide keep/drop.

---

### The 33 orphan columns by source

| Source | Columns | Description |
|--------|---------|-------------|
| **Wyscout player API** | 10 | Player bio/physical data |
| **Transfermarkt scraping** | 6 | Market values, contract info |
| **Competition metadata** | 16 | League info (country, division, dates) |
| **Engineered** | 1 | `transfer_type` (same vs different competition) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=0.85)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

BASE = Path.home() / 'Documents'
def find_file(name):
    r = subprocess.run(['find', str(BASE), '-name', name], capture_output=True, text=True)
    lines = [l for l in r.stdout.strip().split('\n') if l]
    return Path(lines[0]) if lines else None

PARQUET = find_file('transfers_model_2018_2025.parquet')
if PARQUET is None:
    PARQUET = find_file('transfers_model_v2_2018_2025.parquet')
print(f'Loading: {PARQUET}')
df = pd.read_parquet(PARQUET)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]}')

---
## 1. Wyscout Player Bio (10 columns)

Physical and biographical data from the Wyscout/Twelve player API.

In [ ]:
wyscout_cols = ['wy_player_id', 'wyscout_birth_country', 'wyscout_first_name', 
                'wyscout_last_name', 'wyscout_foot', 'wyscout_height', 
                'wyscout_weight', 'wyscout_image_url', 'wyscout_passport', 'wyscout_role']

# Coverage
for c in wyscout_cols:
    null = df[c].isnull().mean() * 100
    uniq = df[c].nunique()
    dtype = df[c].dtype
    print(f'{c:<30} null={null:>5.1f}%  unique={uniq:>6}  dtype={dtype}')

In [ ]:
# For the potentially useful ones: height, weight, foot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Height
ax = axes[0]
h = df['wyscout_height'].dropna()
ax.hist(h, bins=30, color='steelblue', edgecolor='white')
ax.set_title(f'Height (n={len(h):,}, {len(h)/len(df)*100:.1f}% coverage)')
ax.set_xlabel('cm')

# Weight
ax = axes[1]
w = df['wyscout_weight'].dropna()
ax.hist(w, bins=30, color='darkorange', edgecolor='white')
ax.set_title(f'Weight (n={len(w):,}, {len(w)/len(df)*100:.1f}% coverage)')
ax.set_xlabel('kg')

# Foot
ax = axes[2]
f = df['wyscout_foot'].value_counts()
f.plot.bar(ax=ax, color='seagreen', edgecolor='white')
n_foot = df['wyscout_foot'].notna().sum()
ax.set_title(f'Preferred foot (n={n_foot:,}, {n_foot/len(df)*100:.1f}% coverage)')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Passport & birth country — could these be useful as "domestic vs foreign" flag?
print('=== wyscout_birth_country ===')
bc = df['wyscout_birth_country'].value_counts()
print(f'Coverage: {df["wyscout_birth_country"].notna().mean()*100:.1f}%')
print(f'Unique countries: {len(bc)}')
print(f'Top 10:\n{bc.head(10)}')

print('\n=== wyscout_passport ===')
pp = df['wyscout_passport'].dropna()
print(f'Coverage: {len(pp)/len(df)*100:.1f}%')
print(f'Sample values: {pp.head(5).tolist()}')

print('\n=== wyscout_role ===')
print(f'Coverage: {df["wyscout_role"].notna().mean()*100:.1f}%')
print(df['wyscout_role'].value_counts())

## 2. Transfermarkt Data (6 columns)

Market valuation and contract information scraped from Transfermarkt.

In [ ]:
tm_cols = ['tm_player_id', 'tm_transfer_date', 'tm_transfer_value', 
           'tm_transfer_fee', 'tm_remaining_contract_days', 'tm_contract_until_date']

for c in tm_cols:
    null = df[c].isnull().mean() * 100
    uniq = df[c].nunique()
    dtype = df[c].dtype
    print(f'{c:<35} null={null:>5.1f}%  unique={uniq:>6}  dtype={dtype}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Transfer value (log)
ax = axes[0]
v = df['tm_transfer_value'].dropna()
v_pos = v[v > 0]
ax.hist(np.log10(v_pos), bins=50, color='steelblue', edgecolor='white')
ax.set_title(f'log10(transfer_value) — {len(v):,} non-null, {len(v_pos):,} > 0')
ax.set_xlabel('log10(EUR)')

# Transfer fee (log)
ax = axes[1]
f = df['tm_transfer_fee'].dropna()
f_pos = f[f > 0]
if len(f_pos) > 0:
    ax.hist(np.log10(f_pos), bins=50, color='darkorange', edgecolor='white')
ax.set_title(f'log10(transfer_fee) — {len(f):,} non-null, {len(f_pos):,} > 0')
ax.set_xlabel('log10(EUR)')

# Remaining contract days
ax = axes[2]
rcd = df['tm_remaining_contract_days'].dropna()
ax.hist(rcd, bins=50, color='mediumpurple', edgecolor='white')
ax.set_title(f'Remaining contract days — n={len(rcd):,}')
ax.set_xlabel('Days')

plt.tight_layout()
plt.show()

# Quick stats
print('Transfer value:')
print(df['tm_transfer_value'].describe())
print(f'\nZero values: {(df["tm_transfer_value"] == 0).sum():,}')
print(f'\nTransfer fee:')
print(df['tm_transfer_fee'].describe())
print(f'\nRemaining contract days:')
print(df['tm_remaining_contract_days'].describe())

## 3. Competition Metadata (16 columns)

8 columns × 2 (FROM + TO) describing the competition/league.

In [ ]:
comp_cols_from = [c for c in df.columns if c.startswith('from_comp_')]
comp_cols_to = [c for c in df.columns if c.startswith('to_comp_')]

print('=== FROM competition metadata ===')
for c in sorted(comp_cols_from):
    null = df[c].isnull().mean() * 100
    uniq = df[c].nunique()
    print(f'{c:<35} null={null:>5.1f}%  unique={uniq:>5}  dtype={df[c].dtype}')

print('\n=== TO competition metadata ===')
for c in sorted(comp_cols_to):
    null = df[c].isnull().mean() * 100
    uniq = df[c].nunique()
    print(f'{c:<35} null={null:>5.1f}%  unique={uniq:>5}  dtype={df[c].dtype}')

In [ ]:
# Division — this is likely the most useful comp metadata column
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(axes, ['from_comp_division', 'to_comp_division'], ['FROM', 'TO']):
    vals = df[col].value_counts().sort_index()
    vals.plot.bar(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{label} competition division')
    ax.set_xlabel('Division level')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Country — top countries
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, label in zip(axes, ['from_comp_country', 'to_comp_country'], ['FROM', 'TO']):
    top = df[col].value_counts().head(20)
    top.plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'{label} competition country (top 20)')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Season IDs — are they redundant with from_season/to_season?
print('=== Checking if comp_season_id is redundant with season ===')
if 'from_comp_season_id' in df.columns and 'from_season' in df.columns:
    cross = pd.crosstab(df['from_season'], df['from_comp_season_id'])
    print(f'from_season unique: {df["from_season"].nunique()}')
    print(f'from_comp_season_id unique: {df["from_comp_season_id"].nunique()}')
    print(f'\nCross-tab shape: {cross.shape}')
    print('(If season_id maps 1:many to season, it adds info. If 1:1, redundant.)')
    print(f'\nSample mapping:')
    sample = df[['from_season', 'from_comp_season_id']].dropna().drop_duplicates().sort_values('from_season')
    print(f'Unique (season, season_id) pairs: {len(sample)}')
    print(sample.head(20))

## 4. Transfer Type (1 column)

Engineered during processing — indicates whether the transfer was within the same competition or across competitions.

In [ ]:
print(df['transfer_type'].value_counts())
print(f'\nNull: {df["transfer_type"].isnull().sum()}')

fig, ax = plt.subplots(figsize=(6, 4))
df['transfer_type'].value_counts().plot.bar(ax=ax, color=['steelblue', 'darkorange'])
ax.set_title('Transfer type distribution')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 5. Summary — decision framework

**You decide.**

| Column(s) | Coverage | Potential use | Notes |
|-----------|----------|--------------|-------|
| `wy_player_id` | 100% | Row identifier | Essential for tracing results |
| `wyscout_height` | ~23% | Feature (physical) | Low coverage, needs imputation or drop |
| `wyscout_weight` | ~23% | Feature (physical) | Same problem |
| `wyscout_foot` | ~23% | Feature (footedness) | Same problem |
| `wyscout_birth_country` | ~23% | "Domestic/foreign" flag | Needs matching with comp_country |
| `wyscout_passport` | ~23% | Nationality info | Complex multi-passport field |
| `wyscout_role` | ~23% | Position (redundant?) | Already have from_position |
| `wyscout_first/last_name` | ~23% | Display only | No model value |
| `wyscout_image_url` | ~23% | Display only | No model value |
| `tm_player_id` | ? | Cross-reference ID | No model value |
| `tm_transfer_value` | ? | Market value feature | Quality proxy |
| `tm_transfer_fee` | ? | Actual fee paid | Financial context |
| `tm_remaining_contract_days` | ? | Contract leverage | Affects transfer dynamics |
| `tm_transfer_date` | ? | Timing | Could derive window (summer/winter) |
| `tm_contract_until_date` | ? | Contract info | Redundant with remaining_days |
| `from/to_comp_country` | ~70% | Geographic context | League country |
| `from/to_comp_division` | ~70% | League level | Very useful contextual feature |
| `from/to_comp_season_id` | ~70% | Season identifier | Check if redundant with from/to_season |
| `from/to_comp_name` | ~70% | Display only | from/to_competition exists |
| `from/to_comp_completed` | ~70% | Season status | Probably all True |
| `from/to_comp_*_date` | ~70% | Timing | Low model value |
| `transfer_type` | 100% | Binary feature | Same vs different competition |

In [ ]:
# Final coverage summary for all 33 orphan columns
all_orphans = (wyscout_cols + tm_cols + sorted(comp_cols_from) + 
               sorted(comp_cols_to) + ['transfer_type'])

summary = pd.DataFrame({
    'column': all_orphans,
    'null_pct': [df[c].isnull().mean() * 100 for c in all_orphans],
    'nunique': [df[c].nunique() for c in all_orphans],
    'dtype': [str(df[c].dtype) for c in all_orphans],
    'source': (['wyscout'] * len(wyscout_cols) + 
               ['transfermarkt'] * len(tm_cols) + 
               ['comp_meta'] * (len(comp_cols_from) + len(comp_cols_to)) + 
               ['engineered'])
}).sort_values(['source', 'null_pct'])

summary.style.format({'null_pct': '{:.1f}%'}).background_gradient(cmap='YlOrRd', subset=['null_pct'])